# NB7B — Restore metadata + visual semantic audit

Flow: **NB7 error statistics → restore Negative V1 metadata → visual audit → decide**.

- GT tốt → thử diagnosis-aware loss.
- GT noisy/ambiguous → cải thiện negative/localization protocol.
- Không train scorer, không chạy FashionCLIP.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, math
import numpy as np, pandas as pd
EXPECTED_BRANCH="exp/min2-scorer-loo3"; REPO_URL="https://github.com/ThinhTran2208/opisoverated.git"
ROOT=Path("/content/opisoverated")
if not (ROOT/".git").exists():
    subprocess.run(["git","clone","--branch",EXPECTED_BRANCH,"--single-branch",REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"checkout",EXPECTED_BRANCH],check=True)
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only","origin",EXPECTED_BRANCH],check=True)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
print("root",ROOT,"commit",subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"],text=True).strip())

AUDIT_SPLIT="test"; N_CORRECT=10; N_NEAR=15; N_CLEAR=30
NEAR_GAP=.10; CLEAR_GAP=1.0; AUTO_REGENERATE_METADATA=True
EVAL_DIR_OVERRIDE=None; SCORER_READY_DIR_OVERRIDE=None; CORE7_DIR_OVERRIDE=None

from src.data.runtime_paths import load_runtime_paths
from src.data.min2_experiment import scorer_ready_path,metadata_path,category_clean_path,DEFAULT_SEED,prepare_min2_positives,SPLITS
from src.data import build_core7_scorer_dataset as sb
paths=load_runtime_paths(repo_root=ROOT,config_path=ROOT/"configs/data_paths.min2_experiment.json")
MAPPING=ROOT/"configs/category_mapping_core7_v2.json"
try:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
except Exception as e: print("Drive:",e)
MY=Path("/content/drive/MyDrive")
def read_jsonl(p):
    with Path(p).open(encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
def find_file(name):
    for root in (MY/"ML_Final",MY):
        if root.exists():
            hits=list(root.rglob(name))
            if hits: return hits[0]
    return None
def eval_dir():
    if EVAL_DIR_OVERRIDE: return Path(EVAL_DIR_OVERRIDE)
    p=paths.scorer_ready_dir/"evaluation_min2_exp_v1"
    if (p/"loo_predictions_test.jsonl").is_file(): return p
    h=find_file("loo_predictions_test.jsonl")
    if h and (h.parent/"loo_predictions_valid.jsonl").is_file(): return h.parent
    raise FileNotFoundError("Need loo_predictions_valid/test.jsonl from NB6B/NB7 on Drive.")
EVAL=eval_dir(); print("eval",EVAL)
pred={s:read_jsonl(EVAL/f"loo_predictions_{s}.jsonl") for s in ("valid","test")}


In [ ]:
# Restore scorer rows: Tier1 scorer-ready; Tier2 deterministic Negative V1; Tier3 regenerate MIN2 positives/metadata.
split_i={s:i for i,s in enumerate(SPLITS)}
def scorer_name(s): return f"scorer_ready_min2_exp_v1_{s}.jsonl"
def scorer_file(s):
    if SCORER_READY_DIR_OVERRIDE:
        p=Path(SCORER_READY_DIR_OVERRIDE)/scorer_name(s)
        if p.is_file(): return p
    p=scorer_ready_path(paths.scorer_ready_dir,s)
    return p if p.is_file() else find_file(scorer_name(s))
def core_pair(s):
    if CORE7_DIR_OVERRIDE:
        b=Path(CORE7_DIR_OVERRIDE); p=category_clean_path(b,s); m=metadata_path(b,s)
        if p.is_file() and m.is_file(): return p,m
    p=category_clean_path(paths.core7_dir,s); m=metadata_path(paths.core7_dir,s)
    if p.is_file() and m.is_file(): return p,m
    pp=find_file(f"category_clean_{s}.jsonl"); mm=find_file(f"core7_item_metadata_v1_{s}.jsonl")
    if pp and mm and pp.parent==mm.parent: return pp,mm
    return None,None

scorer_rows={}; meta_rows={}; restore={}
for s in ("valid","test"):
    sf=scorer_file(s)
    if sf:
        scorer_rows[s]=read_jsonl(sf); restore[s]=f"scorer-ready:{sf}"
        _,mf=core_pair(s); meta_rows[s]=read_jsonl(mf) if mf else []
        continue
    pf,mf=core_pair(s)
    if (not pf or not mf) and AUTO_REGENERATE_METADATA:
        if not category_clean_path(paths.core7_dir,s).is_file():
            print("Regenerating MIN2 positives/metadata from source...")
            prepare_min2_positives(paths,mapping_path=MAPPING)
        pf,mf=category_clean_path(paths.core7_dir,s),metadata_path(paths.core7_dir,s)
    if not pf or not mf: raise FileNotFoundError(f"Cannot restore metadata split={s}")
    pos,meta=read_jsonl(pf),read_jsonl(mf)
    neg,r=sb.generate_negative_records(pos,meta,split=s,seed=DEFAULT_SEED+split_i[s])
    merged,mr=sb.merge_positive_negative_families(pos,neg)
    assert r["pass"] and mr["pass"]
    scorer_rows[s]=merged; meta_rows[s]=meta; restore[s]=f"reconstructed:{pf}"

negidx={s:{r["sample_id"]:r for r in rows if int(r.get("label",-1))==0} for s,rows in scorer_rows.items()}
for s in ("valid","test"):
    miss=[r["sample_id"] for r in pred[s] if r["sample_id"] not in negidx[s]]
    print(s,"pred",len(pred[s]),"negative",len(negidx[s]),"missing",len(miss),restore[s])
    assert not miss, miss[:10]


In [ ]:
# Enrich LOO predictions with metadata + GT rank/margins.
metaidx={s:{str(r["item_id"]):r for r in rows if r.get("item_id")} for s,rows in meta_rows.items()}
def enrich(r,neg,mi):
    d=[float(x) for x in r["loo_deltas"]]; gt=int(r["gt_swapped_item_index"])
    rank=sorted(range(len(d)),key=lambda i:(-d[i],i)); pr=rank[0]
    nm=neg.get("negative_metadata") or {}; items=[str(x) for x in neg["items"]]
    orig=str(nm.get("original_item_id","")); pos=list(items)
    if orig: pos[gt]=orig
    z=dict(r); z.update(
        gt_rank=rank.index(gt)+1,top1_margin_over_gt=d[pr]-d[gt],gt_delta_nonpositive=d[gt]<=0,
        negative_item_ids=items,positive_item_ids=pos,gt_item_id=items[gt],predicted_item_id=items[pr],
        original_swapped_out_item_id=orig,replacement_item_id=str(nm.get("replacement_item_id","")),
        swap_category=nm.get("swap_category"),ranked_indices=rank,ranked_deltas=[d[i] for i in rank])
    return z
rows=[enrich(r,negidx[AUDIT_SPLIT][r["sample_id"]],metaidx[AUDIT_SPLIT]) for r in pred[AUDIT_SPLIT]]
df=pd.DataFrame(rows)
print("Top1",df.top1_correct.mean(),"Hit@2",df.hit_at_2.mean(),"GT rank>=3",(df.gt_rank>=3).mean(),"GT Δ<=0",(df.gt_delta<=0).mean())
display(df.groupby("outfit_length").agg(n=("sample_id","size"),top1=("top1_correct","mean"),hit2=("hit_at_2","mean"),
    gt_delta_le0=("gt_delta_nonpositive","mean"),mean_gt_delta=("gt_delta","mean")).reset_index())
display(df.groupby("swap_category").agg(n=("sample_id","size"),top1=("top1_correct","mean"),
    gt_delta_le0=("gt_delta_nonpositive","mean"),mean_gap=("top1_margin_over_gt","mean")).reset_index().sort_values("n",ascending=False))


In [ ]:
# Deterministic semantic-audit queue.
A=df[df.gt_rank==1].sample(min(N_CORRECT,(df.gt_rank==1).sum()),random_state=42).assign(audit_group="A_correct")
B0=df[(df.gt_rank==2)&(df.top1_margin_over_gt<=NEAR_GAP)]
B=B0.sample(min(N_NEAR,len(B0)),random_state=42).assign(audit_group="B_near_miss") if len(B0) else B0.assign(audit_group="B_near_miss")
C=df[((df.gt_rank==2)&(df.top1_margin_over_gt>=CLEAR_GAP))|(df.gt_rank>=3)].sort_values(
    ["top1_margin_over_gt","gt_rank"],ascending=False).head(N_CLEAR).assign(audit_group="C_clear_disagreement")
audit=pd.concat([A,B,C],ignore_index=True); print("audit cases",len(audit))
display(audit[["audit_group","sample_id","outfit_length","gt_rank","gt_delta","predicted_top1_delta",
               "top1_margin_over_gt","swap_category","gt_item_id","predicted_item_id"]])


In [ ]:
# Load source images only for audit item IDs. No FashionCLIP inference.
try: from datasets import load_dataset
except ModuleNotFoundError:
    subprocess.run([sys.executable,"-m","pip","install","-q","datasets"],check=True)
    from datasets import load_dataset
need=set()
for _,r in audit.iterrows(): need.update(r.negative_item_ids); need.update(r.positive_item_ids)
items=load_dataset("codewaly/polyvore1000","items",split=AUDIT_SPLIT)
ids=[str(x) for x in items["item_id"]]; ix=[i for i,x in enumerate(ids) if x in need]
images={}
for i in ix:
    q=items[i]; im=q["image"]; images[str(q["item_id"])]=im.convert("RGB") if hasattr(im,"convert") else im
print("needed",len(need),"images",len(images),"missing",len(need-set(images)))


In [ ]:
import matplotlib.pyplot as plt
def cat(item):
    m=metaidx[AUDIT_SPLIT].get(str(item),{})
    return " / ".join(x for x in (m.get("coarse_category"),m.get("master_category")) if x)
def outfit(ids,title,ann=None,deltas=None):
    n=len(ids); cols=min(4,n); rr=math.ceil(n/cols)
    fig,ax=plt.subplots(rr,cols,figsize=(4*cols,4.3*rr)); ax=np.array(ax,dtype=object).reshape(-1)
    for i,a in enumerate(ax):
        a.axis("off")
        if i>=n: continue
        x=str(ids[i]); im=images.get(x)
        if im is not None:a.imshow(im)
        tag=(ann or {}).get(i,""); dt=f"\nΔ={float(deltas[i]):+.3f}" if deltas is not None else ""
        a.set_title(f"idx={i} {tag}\n{x}\n{cat(x)}{dt}",fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()
def show_case(r):
    if isinstance(r,pd.Series): r=r.to_dict()
    gt=int(r["gt_swapped_item_index"]); pr=int(r["predicted_problematic_index"])
    print("\n",r["audit_group"],r["sample_id"],"GT rank",r["gt_rank"],"GT Δ",round(r["gt_delta"],3),
          "Pred Δ",round(r["predicted_top1_delta"],3),"gap",round(r["top1_margin_over_gt"],3),
          "swap",r["swap_category"],"GT",r["gt_item_id"],"PRED",r["predicted_item_id"])
    outfit(r["positive_item_ids"],"ORIGINAL POSITIVE")
    ann={i:"["+" + ".join(t for t in (("GT/SWAP" if i==gt else ""),("PRED" if i==pr else "")) if t)+"]"
         for i in range(len(r["negative_item_ids"]))}
    outfit(r["negative_item_ids"],"SYNTHETIC NEGATIVE",ann,r["loo_deltas"])
for _,r in audit[audit.audit_group=="C_clear_disagreement"].iterrows(): show_case(r)


In [ ]:
# Human labels persisted to Drive.
LABELS=["GT_CLEARLY_PROBLEMATIC","PRED_CLEARLY_PROBLEMATIC","BOTH_PLAUSIBLE_AMBIGUOUS","DATA_OR_IMAGE_ISSUE"]
OUT=(MY/"ML_Final/min2_exp_v1/loo_semantic_audit") if MY.exists() else paths.scorer_ready_dir/"loo_semantic_audit"
OUT.mkdir(parents=True,exist_ok=True); LP=OUT/f"semantic_audit_labels_{AUDIT_SPLIT}.csv"
if not LP.is_file():
    pd.DataFrame({"sample_id":audit.sample_id,"audit_group":audit.audit_group,"semantic_label":"","notes":""}).to_csv(LP,index=False)
print("labels file:",LP)
print("After viewing images, fill JUDGMENTS and rerun this cell.")
JUDGMENTS={
 # "sample_id": ("GT_CLEARLY_PROBLEMATIC","note"),
}
lab=pd.read_csv(LP).fillna("").set_index("sample_id")
for sid,val in JUDGMENTS.items():
    label,note=(val,"") if isinstance(val,str) else val
    assert label in LABELS
    lab.loc[sid,"semantic_label"]=label; lab.loc[sid,"notes"]=note
lab.reset_index().to_csv(LP,index=False)
display(lab.reset_index())


In [ ]:
# Decision summary. Heuristic only; not an official benchmark acceptance criterion.
lab=pd.read_csv(LP).fillna("")
j=audit.merge(lab[["sample_id","semantic_label","notes"]],on="sample_id",how="left")
cl=j[(j.audit_group=="C_clear_disagreement")&j.semantic_label.isin(LABELS)]
cnt=cl.semantic_label.value_counts(); n=len(cl)
frac=lambda x: float(cnt.get(x,0)/n) if n else None
rec="INSUFFICIENT_MANUAL_AUDIT"
if n>=20:
    gt=frac("GT_CLEARLY_PROBLEMATIC") or 0; pr=frac("PRED_CLEARLY_PROBLEMATIC") or 0; amb=frac("BOTH_PLAUSIBLE_AMBIGUOUS") or 0
    if gt>=.70 and amb<=.20: rec="GROUND_TRUTH_LOOKS_USABLE__TRY_DIAGNOSIS_AWARE_LOSS"
    elif pr+amb>=.50 or gt<.50: rec="GROUND_TRUTH_LOOKS_NOISY__IMPROVE_NEGATIVE_LOCALIZATION_PROTOCOL"
    else: rec="MIXED__AUDIT_MORE_CASES"
report={"split":AUDIT_SPLIT,"automated":{"n":len(df),"top1":float(df.top1_correct.mean()),"hit2":float(df.hit_at_2.mean()),
        "gt_rank_ge3":float((df.gt_rank>=3).mean()),"gt_delta_le0":float((df.gt_delta<=0).mean())},
        "clear_manual_n":n,"clear_label_distribution":{x:{"count":int(cnt.get(x,0)),"fraction":frac(x)} for x in LABELS},
        "recommendation":rec,"restore_mode":restore}
rp=OUT/f"semantic_audit_report_{AUDIT_SPLIT}.json"; rp.write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(report,ensure_ascii=False,indent=2)); print("saved",rp)
